In [87]:
import pickle 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import json


In [88]:
def calcular_score(probabilidad, target_score = 650, target_odds = 50, pdo = 20):
    #evitar log(0) o division por cero
    probabilidad = np.clip(probabilidad, 0.0001, 0.9999)
    #Calcular los odds(probabilidad de no default / probabilidad de default)
    #Como el modelo predice Default, la probabilidad de no default es 1 - probabilidad
    odds = (1 - probabilidad) / probabilidad
    #Calcular factores de escala y offset
    factor = pdo / np.log(2)
    offset = target_score - (factor * np.log(target_odds))
    #Formula final del scorecard
    score = offset + (factor * np.log(odds))
    return int(score)
#Cabe destacar que estoy usando valores convecionales para el scorecard, pero estos pueden ser ajustados según las necesidades del negocio o la interpretación de los resultados.

In [89]:
#Aqui vuelvo a cargar la red neuronal, pero esta vez para usarla en la función de scorecard
class CreditRiskNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            # Capa 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            # Capa 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            # Capa 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            # Salida — probabilidad de default
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


In [90]:

# Cargar el esquema para obtener las columnas exactas
with open('esquema_modelo.json', 'r') as f:
    esquema = json.load(f)

cols = esquema['columnas_modelo']

# Crear un DataFrame vacío con las columnas del modelo
df_test = pd.DataFrame(columns=cols)

# Función para crear un usuario base con valores en cero
def crear_usuario_vacio():
    return {col: 0.0 for col in cols}

usuarios = []

# --- USUARIO 1: Muy Bajo Riesgo (El ideal) ---
u1 = crear_usuario_vacio()
u1.update({
    'loan_amnt': 10000, 'funded_amnt': 10000, 'term': 36, 'int_rate': 7.5, 'installment': 311,
    'emp_length': 10, 'annual_inc': 120000, 'dti': 5.0, 'revol_util': 15.0, 'total_acc': 20,
    'home_ownership_MORTGAGE': 1, 'verification_status_Verified': 1, 'purpose_credit_card': 1,
    'addr_region_West': 1, 'emp_title_cat_Management': 1
})
usuarios.append(u1)

# --- USUARIO 2: Muy Alto Riesgo (Default probable) ---
u2 = crear_usuario_vacio()
u2.update({
    'loan_amnt': 35000, 'funded_amnt': 35000, 'term': 60, 'int_rate': 24.0, 'installment': 1000,
    'emp_length': 1, 'annual_inc': 25000, 'dti': 35.0, 'delinq_2yrs': 2, 'inq_last_6mths': 5,
    'revol_util': 95.0, 'home_ownership_RENT': 1, 'purpose_small_business': 1,
    'addr_region_South': 1, 'emp_title_cat_Other': 1
})
usuarios.append(u2)

# --- USUARIO 3: Riesgo Medio (Perfil común) ---
u3 = crear_usuario_vacio()
u3.update({
    'loan_amnt': 15000, 'funded_amnt': 15000, 'term': 36, 'int_rate': 12.0, 'annual_inc': 55000,
    'dti': 18.0, 'open_acc': 10, 'revol_bal': 12000, 'home_ownership_RENT': 1, 
    'purpose_debt_consolidation': 1, 'addr_region_Northeast': 1, 'emp_title_cat_Healthcare': 1
})
usuarios.append(u3)

# --- USUARIO 4: Profesional Consolidado ---
u4 = crear_usuario_vacio()
u4.update({
    'loan_amnt': 20000, 'term': 36, 'int_rate': 8.5, 'annual_inc': 95000, 'dti': 12.0,
    'home_ownership_OWN': 1, 'purpose_home_improvement': 1, 'addr_region_West': 1, 'emp_title_cat_Tech': 1
})
usuarios.append(u4)

# --- USUARIO 5: Riesgo por sobre-endeudamiento ---
u5 = crear_usuario_vacio()
u5.update({
    'loan_amnt': 25000, 'term': 60, 'int_rate': 18.0, 'annual_inc': 45000, 'dti': 40.0,
    'revol_util': 88.0, 'home_ownership_RENT': 1, 'purpose_other': 1
})
usuarios.append(u5)

# --- USUARIO 6: Solicitud Conjunta (Estable) ---
u6 = crear_usuario_vacio()
u6.update({
    'loan_amnt': 30000, 'term': 60, 'int_rate': 11.0, 'annual_inc': 150000, 'dti': 10.0,
    'application_type_JOINT': 1, 'home_ownership_MORTGAGE': 1, 'purpose_house': 1, 'addr_region_South': 1
})
usuarios.append(u6)

# --- USUARIO 7: Recién graduado (Incierto) ---
u7 = crear_usuario_vacio()
u7.update({
    'loan_amnt': 5000, 'term': 36, 'int_rate': 14.0, 'annual_inc': 35000, 'emp_length': 0,
    'dti': 22.0, 'home_ownership_RENT': 1, 'purpose_educational': 1, 'addr_region_West': 1
})
usuarios.append(u7)

# --- USUARIO 8: Perfil con historial de morosidad antigua ---
u8 = crear_usuario_vacio()
u8.update({
    'loan_amnt': 12000, 'term': 36, 'int_rate': 16.0, 'annual_inc': 60000, 'mths_since_last_delinq': 48,
    'pub_rec': 1, 'home_ownership_MORTGAGE': 1, 'purpose_debt_consolidation': 1, 'addr_region_South': 1
})
usuarios.append(u8)

# --- USUARIO 9: Pequeño préstamo / Gran ingreso ---
u9 = crear_usuario_vacio()
u9.update({
    'loan_amnt': 2000, 'term': 36, 'int_rate': 6.0, 'annual_inc': 200000, 'dti': 1.0,
    'home_ownership_OWN': 1, 'purpose_major_purchase': 1, 'addr_region_West': 1
})
usuarios.append(u9)

# --- USUARIO 10: Riesgo Geográfico/Sectorial ---
u10 = crear_usuario_vacio()
u10.update({
    'loan_amnt': 18000, 'term': 60, 'int_rate': 19.5, 'annual_inc': 48000, 'dti': 28.0,
    'home_ownership_RENT': 1, 'purpose_medical': 1, 'emp_title_cat_Self_Employed': 1
})
usuarios.append(u10)

df_final = pd.DataFrame(usuarios)
df_final.to_csv('usuarios_prueba_modelo.csv', index=False)

In [91]:
#Cargar el Scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

#Cargar la red neuronal
input_size = len(scaler.mean_)

model = CreditRiskNN(input_dim=input_size)
model.load_state_dict(torch.load('credit_risk_nn.pth'))
model.eval()


CreditRiskNN(
  (network): Sequential(
    (0): Linear(in_features=68, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
    (13): Sigmoid()
  )
)

In [92]:
datos_clientes = pd.read_csv('usuarios_prueba_modelo.csv')


for i in range(len(datos_clientes)):
    cliente = datos_clientes.iloc[i:i+1]  # Seleccionar una fila como DataFrame
    
    datos_clientes_scaled = scaler.transform(cliente)
    datos_tensor = torch.FloatTensor(datos_clientes_scaled)
    
    with torch.no_grad():
        prediccion = model(datos_tensor).item()

    puntos = calcular_score(prediccion)
    print(f"Cliente {i+1} - Probabilidad de default: {prediccion:.4f} - Scorecard: {puntos}")
    print('\n')

Cliente 1 - Probabilidad de default: 0.0000 - Scorecard: 802


Cliente 2 - Probabilidad de default: 1.0000 - Scorecard: 271


Cliente 3 - Probabilidad de default: 0.0011 - Scorecard: 734


Cliente 4 - Probabilidad de default: 0.0000 - Scorecard: 802


Cliente 5 - Probabilidad de default: 0.9992 - Scorecard: 331


Cliente 6 - Probabilidad de default: 1.0000 - Scorecard: 271


Cliente 7 - Probabilidad de default: 0.0053 - Scorecard: 687


Cliente 8 - Probabilidad de default: 0.0002 - Scorecard: 790


Cliente 9 - Probabilidad de default: 0.0000 - Scorecard: 802


Cliente 10 - Probabilidad de default: 0.4265 - Scorecard: 545




c:\Users\Jhanuar\Desktop\Universidad\8vo semestre\RNA y Algo. Bioinsp\Trabajo 2\RNA-Trabajo-2-main\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Jhanuar\Desktop\Universidad\8vo semestre\RNA y Algo. Bioinsp\Trabajo 2\RNA-Trabajo-2-main\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Jhanuar\Desktop\Universidad\8vo semestre\RNA y Algo. Bioinsp\Trabajo 2\RNA-Trabajo-2-main\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\Jhanuar\Desktop\Universidad\8vo semestre\RNA y Algo. Bioinsp\Trabajo 2\RNA-Trabajo-2-main\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted